# LSTM Forecast

Train a univariate LSTM and recursively forecast the shared test window without using test-period observations.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.layers import Dense, Dropout, Input, LSTM
from tensorflow.keras.models import Sequential

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"


In [ ]:
stock_data = pd.read_csv(DATA_DIR / "tcs_stock_data_cleaned.csv", parse_dates=["Date"])
stock_data = stock_data.sort_values("Date").reset_index(drop=True)
split_index = int(len(stock_data) * 0.8)
train = stock_data.iloc[:split_index]
test = stock_data.iloc[split_index:].copy()

scaler = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(train[["Close"]])


In [ ]:
def create_sequences(values: np.ndarray, steps: int) -> tuple[np.ndarray, np.ndarray]:
    inputs, targets = [], []
    for index in range(steps, len(values)):
        inputs.append(values[index - steps:index, 0])
        targets.append(values[index, 0])
    return np.asarray(inputs), np.asarray(targets)


time_steps = 60
X_train, y_train = create_sequences(train_scaled, time_steps)
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))


In [ ]:
model = Sequential(
    [
        Input(shape=(time_steps, 1)),
        LSTM(50, return_sequences=True),
        Dropout(0.2),
        LSTM(50),
        Dropout(0.2),
        Dense(25),
        Dense(1),
    ]
)
model.compile(optimizer="adam", loss="mean_squared_error")
history = model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.1,
    shuffle=False,
    verbose=1,
)


In [ ]:
window = train_scaled[-time_steps:, 0].copy()
scaled_forecast = []
for _ in range(len(test)):
    next_value = float(model.predict(window.reshape(1, time_steps, 1), verbose=0)[0, 0])
    scaled_forecast.append(next_value)
    window = np.append(window[1:], next_value)

forecast = scaler.inverse_transform(np.asarray(scaled_forecast).reshape(-1, 1)).ravel()
results = pd.DataFrame(
    {
        "Date": test["Date"],
        "Actual": test["Close"].to_numpy(),
        "LSTM_Prediction": forecast,
    }
)


In [ ]:
mae = mean_absolute_error(results["Actual"], results["LSTM_Prediction"])
rmse = np.sqrt(mean_squared_error(results["Actual"], results["LSTM_Prediction"]))
pd.Series({"MAE": mae, "RMSE": rmse})


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(results["Date"], results["Actual"], label="Actual")
ax.plot(results["Date"], results["LSTM_Prediction"], label="LSTM")
ax.set(title="LSTM Forecast vs Actual", xlabel="Date", ylabel="Closing Price (INR)")
ax.legend()
fig.autofmt_xdate()
plt.show()


In [ ]:
results.to_csv(DATA_DIR / "lstm_predictions.csv", index=False)
model.save(MODEL_DIR / "lstm_model.keras")
